## Packages & Data

In [ ]:
library(tidyverse)
library(lme4)
library(lmerTest)
library(broom)
library(broom.mixed)
library(ggeffects)
library(emmeans)
library(performance)
library(patchwork)
library(cowplot)
library(modelsummary)
library(datawizard)
library(see)
# source("scripts/simulate_lmm_data.R")

In [ ]:
simulate_lmm_data <- function() {
 
set.seed(123)
 
n_subjects <- 30
n_time <- 6
 
subject_df <- tibble(
  subject = factor(1:n_subjects),
  b0 = rnorm(n_subjects, mean = 0, sd = 6),
  b1 = rnorm(n_subjects, mean = 0, sd = 1.2)
)
 
df <- expand_grid(
  subject = factor(1:n_subjects),
  time = 0:(n_time - 1)
) |>
  left_join(subject_df, by = "subject") |>
  mutate(
    y = 20 + 2.5 * time + b0 + b1 * time + rnorm(n(), mean = 0, sd = 2)
  )
  return(df)
  }

In [ ]:
#  Organize Function (equivalent of dplyr's arrange function). From https://stackoverflow.com/questions/75667332/base-analogue-of-arrange-in-pipelines
organize <- function (df, ..., na.last = TRUE, decreasing = FALSE,
                      method = c("auto", "shell", "radix"), drop = FALSE) {
  method <- match.arg(method)
  dots <- eval(substitute(alist(...)))
  exprs <- lapply(dots, FUN = \(e) eval(e, df, parent.frame())) 
  arg_ls <- c(exprs,
              na.last = na.last,
              decreasing = decreasing,
              method = method
              )
  df[do.call("order", arg_ls), , drop = drop]
}

In [ ]:
tbl1 <- read.csv("036_ADNI Merge Modelling Cohort")

## BMI

### BMI & Biomarkers

In [ ]:
# Scaling BMI and looking at influence on TAU
tbl1$BMI_z <- scale(tbl1$BMI)
m1 <- lmer(TAU_capped ~ VISCODEnum_capped * BMI_capped + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m1)

In [ ]:
# Scaling BMI and looking at influence on ABETA
m2 <- lmer(ABETA_capped ~ VISCODEnum_capped * BMI_capped + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m2)

In [ ]:
# Scaling BMI and looking at influence on PTAU
m3 <- lmer(PTAU_capped ~ VISCODEnum_capped * BMI_capped + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m3)

### BMI & IR

In [ ]:
# Pnt Baselines
tbl2 <- tbl1 |>
    subset(VISCODE == "bl")

In [ ]:
# Linear Model - BMI x METS-IR
lm1 <- lm(METSIR_capped ~ BMI_capped, data = tbl2)
summary(lm1)

In [ ]:
# Linear Model - BMI x TYG
lm2 <- lm(TYG_capped ~ BMI_capped, data = tbl2)
summary(lm2)

In [ ]:
# Linear Model - BMI x TYG-BMI
lm3 <- lm(TYG_BMI_capped ~ BMI_capped, data = tbl2)
summary(lm3)

In [ ]:
# Linear Model - BMI x TG/HDL
lm4 <- lm(TG_HDL_capped ~ BMI_capped, data = tbl2)
summary(lm4)

## Age


### Age & Biomarkers

In [ ]:
# Age's influence at influence on TAU
m4 <- lmer(TAU_capped ~ VISCODEnum_capped * AGE_capped + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m4)

In [ ]:
# Age's influence at influence on ABETA
m5 <- lmer(ABETA_capped ~ VISCODEnum_capped * AGE_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m5)

In [ ]:
# Age's influence at influence on PTAU
m6 <- lmer(PTAU_capped ~ VISCODEnum_capped * AGE_capped + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m6)

### Age & IR

In [ ]:
# Linear Model - Age x METS-IR
lm5 <- lm(METSIR_capped ~ AGE_capped, data = tbl2)
summary(lm5)
# Increased age correlated to decreased IR

In [ ]:
# Linear Model - Age x TYG
lm6 <- lm(TYG_capped ~ AGE_capped, data = tbl2)
summary(lm6)
# Non-significant

In [ ]:
# Linear Model - Age x TYG-BMI
lm7 <- lm(TYG_BMI_capped ~ AGE_capped, data = tbl2)
summary(lm7)
# Increased age correlated to decreased IR

In [ ]:
# Linear Model - Age x TYG-BMI
lm8 <- lm(TG_HDL_capped ~ AGE_capped, data = tbl2)
summary(lm8)
# Non-significant

## Sex

### Sex & Biomarkers

In [ ]:
# Tau x Pt Gender
m7 <- lmer(TAU_capped ~ VISCODEnum_capped * PTGENDER + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m7)
# Male = less Tau

In [ ]:
# ABETA x Pt Gender
m8 <- lmer(ABETA_capped ~ VISCODEnum_capped * PTGENDER + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m8)
# Non-significant

In [ ]:
# PTAU x Pt Gender
m9 <- lmer(PTAU_capped ~ VISCODEnum_capped * PTGENDER + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m9)
# Male = less PTAU

### Sex & IR

In [ ]:
# Linear Model - Sex x METS-IR
lm9 <- lm(METSIR_z ~ PTGENDER, data = tbl2)
summary(lm9)
# Male linked to higher IR

In [ ]:
# Linear Model - Sex x TyG
lm10 <- lm(TYG_z ~ PTGENDER, data = tbl2)
summary(lm10)
# Non-significant

In [ ]:
# Linear Model - Sex x TyG-BMI
lm11 <- lm(TYG_BMI_z ~ PTGENDER, data = tbl2)
summary(lm11)
# Non-significant

In [ ]:
# Linear Model - Sex x TG/HDL 
lm12 <- lm(TG_HDL_z ~ PTGENDER, data = tbl2)
summary(lm12)
# Male linked to higher IR  

## ApoE E4

### ApoE & Biomarkers

In [ ]:
# APOE's influence at influence on TAU
m10 <- lmer(TAU_capped ~ VISCODEnum_capped * APOE4 + (VISCODEnum_capped| RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m10)

In [ ]:
# APOE's influence at influence on ABETA
m11 <- lmer(ABETA_capped ~ VISCODEnum_capped * APOE4 + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m11)

In [ ]:
# APOE's influence at influence on PTAU
m12 <- lmer(PTAU_capped ~ VISCODEnum_capped * APOE4 + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m12)

### APOE & IR

In [ ]:
# APOE & METS-IR
lm13 <- lm(METSIR_z ~ APOE4, data = tbl2)
summary(lm13)
# APOE sees decreased IR

In [ ]:
# APOE & TYG
lm14 <- lm(TYG_z ~ APOE4, data = tbl2)
summary(lm14)
# Non-significant

In [ ]:
# APOE & TYG-BMI
lm15 <- lm(TYG_BMI_z ~ APOE4, data = tbl2)
summary(lm15)
# APOE leads to decreased IR

In [ ]:
# APOE & TG/HDL
lm16 <- lm(TG_HDL_z ~ APOE4, data = tbl2)
summary(lm16)
# Non-significant

## Education

### Education & Biomarkers

In [ ]:
# Education & Tau
m13 <- lmer(TAU_capped ~ VISCODEnum_capped * PTEDUCAT + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m13)
# Greater education saw decreased tau (protective)

In [ ]:
# Education & ABETA
m14 <- lmer(ABETA_capped ~ VISCODEnum_capped * PTEDUCAT + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m14)
# Non-significant

In [ ]:
# Education & PTAU
m15 <- lmer(PTAU_capped ~ VISCODEnum_capped * PTEDUCAT + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m15)
# Greater education saw decreased pTAU

### Education & IR

In [ ]:
# Education & METS-IR
lm17 <- lm(METSIR_z ~ PTEDUCAT, data = tbl2)
summary(lm17)
# Non-significant

In [ ]:
# Education & TYG
lm18 <- lm(TYG_z ~ PTEDUCAT, data = tbl2)
summary(lm18)
# Education leads to slightly less IR (significant)

In [ ]:
# Education & TYG-BMI
lm19 <- lm(TYG_BMI_z ~ PTEDUCAT, data = tbl2)
summary(lm19)
# Non-significant

In [ ]:
# Education & TG/HDL
lm20 <- lm(TG_HDL_z ~ PTEDUCAT, data = tbl2)
summary(lm20)
# Non-significant

## Diagnosis

In [ ]:
table(tbl1$DX_bl)

In [ ]:
# Grouping baseline diagnosis such that CN + SMC = CN and EMCI + LMCI = MCI
tbl3 <- tbl1 |>
    mutate(DIAGNOSIS_bl = if_else(DX_bl == "AD", "AD", 
                                  if_else(DX_bl == "EMCI", "MCI", 
                                  if_else(DX_bl == "LMCI", "MCI", 
                                  if_else(DX_bl == "SMC", "CN", "CN")))))
table(tbl3$DIAGNOSIS_bl)

In [ ]:
write.csv(tbl3, "047_ADNI Merge Modelling Cohort V2")

In [ ]:
# Creating Baseline-only table
tbl4 <- tbl3 |>
    subset(VISCODE == "bl")

### Diagnosis @ Bl and Biomarkers

In [ ]:
# Bl Diagnosis & Tau
m16 <- lmer(TAU_capped ~ VISCODEnum_capped * DIAGNOSIS_bl + (VISCODEnum_capped | RID), data = tbl3, control = lmerControl(optimizer = "bobyqa"))
summary(m16)
# CN and MCI have significantly less TAU than AD

In [ ]:
# Bl Diagnosis & ABETA
m17 <- lmer(ABETA_capped ~ VISCODEnum_capped * DIAGNOSIS_bl + (VISCODEnum_capped | RID), data = tbl3, control = lmerControl(optimizer = "bobyqa"))
summary(m17)
# CN and MCI have significantly more ABETA than AD

In [ ]:
# Bl Diagnosis & PTAU
m18 <- lmer(PTAU_capped ~ VISCODEnum_capped * DIAGNOSIS_bl + (VISCODEnum_capped | RID), data = tbl3, control = lmerControl(optimizer = "bobyqa"))
summary(m18)
# CN and MCI have significantly less pTAU than AD

### Bl Diagnosis & IR

In [ ]:
# Diagnosis & METS-IR
lm21 <- lm(METSIR_z ~ DIAGNOSIS_bl, data = tbl4)
summary(lm21)
# AD < CN < MCI group = hierarchy of IR

In [ ]:
# Diagnosis & TYG
lm22 <- lm(TYG_z ~ DIAGNOSIS_bl, data = tbl4)
summary(lm22)
# Non-significant

In [ ]:
# Diagnosis & TYG-BMI
lm23 <- lm(TYG_BMI_z ~ DIAGNOSIS_bl, data = tbl4)
summary(lm23)
# AD < CN < MCI group = hierarchy of IR

In [ ]:
# Diagnosis & TG/HDL
lm24 <- lm(TG_HDL_z ~ DIAGNOSIS_bl, data = tbl4)
summary(lm24)
# Non-significant

## Diabetes

### Diabetes & Biomarkers

In [ ]:
# Diabetes & Tau
m19 <- lmer(TAU_capped ~ VISCODEnum_capped * DIABETES + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m19)
# Non-significant

In [ ]:
# Diabetes & ABETA
m20 <- lmer(ABETA_capped ~ VISCODEnum_capped * DIABETES + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m20)
# Non-significant

In [ ]:
# Diabetes & PTAU
m21 <- lmer(PTAU_capped ~ VISCODEnum_capped * DIABETES + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m21)
# Non-significant

### Diabetes & IR

In [ ]:
# Diabetes & METS-IR
lm25 <- lm(METSIR_z ~ DIABETES, data = tbl2)
summary(lm25)
# As expected, DIABETES leads to a significantly higher level of IR

In [ ]:
# Diabetes & TYG
lm26 <- lm(TYG_z ~ DIABETES, data = tbl2)
summary(lm26)
# As expected, DIABETES leads to a significantly higher level of IR

In [ ]:
# Diabetes & TYG-BMI
lm27 <- lm(TYG_BMI_z ~ DIABETES, data = tbl2)
summary(lm27)
# As expected, DIABETES leads to a significantly higher level of IR

In [ ]:
# Diabetes & TYG-BMI
lm28 <- lm(TG_HDL_z ~ DIABETES, data = tbl2)
summary(lm28)
# As expected, DIABETES leads to a significantly higher level of IR

In [ ]:
# Diabetes demonstrates greatest association with TyG, followed by METS-IR, TyG-BMI, and far behind, TG/HDL

## Hypertension

### Hypertension & Biomarkers

In [ ]:
# Hypertension & Tau
m22 <- lmer(TAU_capped ~ VISCODEnum_capped * HYPERTENSION + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m22)
# Non-significant

In [ ]:
# Hypertension & ABETA
m23 <- lmer(ABETA_capped ~ VISCODEnum_capped * HYPERTENSION + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m23)
# Non-significant

In [ ]:
# Hypertension & PTAU
m24 <- lmer(PTAU_capped ~ VISCODEnum_capped * HYPERTENSION + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m24)
# Non-significant

### Hypertension & IR

In [ ]:
# Hypertension & METS-IR
lm29 <- lm(METSIR_z ~ HYPERTENSION, data = tbl2)
summary(lm29)
# Non-significant

In [ ]:
# Hypertension & TYG
lm30 <- lm(TYG_z ~ HYPERTENSION, data = tbl2)
summary(lm30)
# Non-significant

In [ ]:
# Hypertension & TYG-BMI
lm31 <- lm(TYG_BMI_z ~ HYPERTENSION, data = tbl2)
summary(lm31)
# Non-significant

In [ ]:
# Hypertension & TG/HDL
lm32 <- lm(TG_HDL_z ~ HYPERTENSION, data = tbl2)
summary(lm32)
# Non-significant

## Smoking 

### Smoking & Biomarkers

In [ ]:
# Smoking & Tau
m25 <- lmer(TAU_capped ~ VISCODEnum_capped * SMOKING + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m25)
# Non-significant

In [ ]:
# Smoking & ABETA
m26 <- lmer(ABETA_capped ~ VISCODEnum_capped * SMOKING + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m26)
# Non-significant

In [ ]:
# Smoking & PTAU
m27 <- lmer(PTAU_capped ~ VISCODEnum_capped * SMOKING + (VISCODEnum_capped | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m27)
# Non-significant

### Smoking & IR

In [ ]:
# Smoking & METS-IR
lm33 <- lm(METSIR_z ~ SMOKING, data = tbl2)
summary(lm33)
# Non-significant

In [ ]:
# Smoking & TYG
lm34 <- lm(TYG_z ~ SMOKING, data = tbl2)
summary(lm34)
# Non-significant

In [ ]:
# Smoking & TYG-BMI
lm35 <- lm(TYG_BMI_z ~ SMOKING, data = tbl2)
summary(lm35)
# Non-significant

In [ ]:
# Smoking & TG/HDL
lm36 <- lm(TG_HDL_z ~ SMOKING, data = tbl2)
summary(lm36)
# Non-significant

## Race

### Race & Biomarkers

In [ ]:
# Race & Tau
m28 <- lmer(TAU ~ VISCODEnum_z * PTRACCAT + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m28)
# Non-significant

In [ ]:
# Race & ABETA
m29 <- lmer(ABETA_clean ~ VISCODEnum_z * PTRACCAT + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m29)
# Non-significant

In [ ]:
# Race & PTAU
m30 <- lmer(PTAU_clean ~ VISCODEnum_z * PTRACCAT + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(m30)
# Non-significant